# 02. 스타일 × 향조 소프트 매핑 행렬

Kobayashi Color Image Scale(스타일 좌표)과 Edwards Fragrance Wheel·Zarzo & Stanton PCA(향조 좌표)를 같은 2D 평면에 놓고,
유클리드 거리 → `exp(-d)` → 행 정규화로 6 × 5 매핑 행렬을 만듭니다. 결과: [`assets/style_accord_mapping.json`](../assets/style_accord_mapping.json)


In [4]:
"""
2D 좌표 기반 패션 스타일-향조 소프트 매핑
근거:
  - 스타일 좌표: Kobayashi Color Image Scale (1990) → Cool/Warm(X축), Soft/Hard(Y축)
  - 향조 좌표: Michael Edwards Fragrance Wheel (2023) 원형 구조
              + Zarzo & Stanton (2009) 향수 지각 PCA 2축
              → X축: Cool(Fresh/Woody) ↔ Warm(Floral/Amber)
              → Y축: Light(-y) ↔ Heavy(+y)

향수 휠 각도 매핑 원리:
  Edwards 휠의 4대 계열을 4사분면에 배치:
    Floral  (315°) → Warm + Light  → (+x, -y)
    Amber   ( 45°) → Warm + Heavy  → (+x, +y)
    Woody   (135°) → Cool + Heavy  → (-x, +y)
    Fresh   (225°) → Cool + Light  → (-x, -y)

  서브계열 위치 기반 향조 각도:
    floral   → Floral 계열 중심           → 315°
    intense  → Amber(Oriental) 계열       →  45°
    natural  → Fresh 계열                 → 225°
    niche    → Dry Woods (Woody~Fresh 경계) → 157°
    romantic → Soft Floral (Floral~Amber 경계) → 337°
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings

warnings.filterwarnings("ignore")

# ── 1. 데이터 로드 ─────────────────────────────────────────────────────────────
df = pd.read_csv('styles_classified.csv')
df = df[df['style_class'] != 'Unknown'].copy()
print(f"총 샘플 수: {len(df)}")
print(f"스타일 분포:\n{df['style_class'].value_counts()}\n")

# ── 2. 스타일 좌표 (Kobayashi Color Image Scale 기반) ──────────────────────────
# X축: Cool(-1) ↔ Warm(+1)
# Y축: Light(-1) ↔ Heavy(+1)
STYLE_COORDS = {
    'Romantic': ( 0.40, -0.60),   # Warm + Light: Pink/Lavender 계열
    'Casual':   (-0.20, -0.40),   # Slightly Cool + Light: Beige/Cream
    'Sporty':   (-0.60, -0.50),   # Cool + Light: White/Bright Blue
    'Formal':   ( 0.40,  0.60),   # Warm + Heavy: Navy/Burgundy
    'Street':   (-0.50,  0.40),   # Cool + Heavy: Black/Dark Grey
    'Vintage':  ( 0.50,  0.50),   # Warm + Heavy: Brown/Mustard/Rust
}

# ── 3. 향조 좌표 (Edwards 향수 휠 → x=r·cos(θ), y=r·sin(θ)) ──────────────────
ACCORD_WHEEL = {
    'floral':   315,   # Floral 계열 중심
    'intense':   45,   # Amber(Oriental) 계열
    'natural':  225,   # Fresh 계열
    'niche':    157,   # Dry Woods (Woody 135° ~ Fresh 225° 사이)
    'romantic': 337,   # Soft Floral (Floral 315° ~ Amber 45° 사이)
}

r = 0.80  # 반경 (스타일 좌표와 동일 스케일)

ACCORD_COORDS = {
    accord: (
        round(r * np.cos(np.radians(angle)), 4),
        round(r * np.sin(np.radians(angle)), 4)
    )
    for accord, angle in ACCORD_WHEEL.items()
}

print("=== 향조 좌표 (Edwards 향수 휠 각도 → 직교 좌표) ===")
print(f"{'향조':10s}  {'각도':>6s}  {'x':>7s}  {'y':>7s}")
for accord, angle in ACCORD_WHEEL.items():
    x, y = ACCORD_COORDS[accord]
    print(f"{accord:10s}  {angle:>5}°  {x:>7.4f}  {y:>7.4f}")
print()

# ── 4. 스타일-향조 유클리드 거리 계산 ─────────────────────────────────────────
styles  = list(STYLE_COORDS.keys())
accords = list(ACCORD_COORDS.keys())

dist_data = {}
for style in styles:
    sx, sy = STYLE_COORDS[style]
    row = {}
    for accord in accords:
        ax, ay = ACCORD_COORDS[accord]
        row[accord] = np.sqrt((sx - ax)**2 + (sy - ay)**2)
    dist_data[style] = row

dist_df = pd.DataFrame(dist_data).T   # (6 styles × 5 accords)

print("=== 스타일-향조 유클리드 거리 ===")
print(dist_df.round(4))
print()

# ── 5. 거리 → 유사도 변환 (exp(-d)) → 행 정규화 ───────────────────────────────
sim_matrix = np.exp(-dist_df.values.astype(float))
sim_norm   = sim_matrix / sim_matrix.sum(axis=1, keepdims=True)

mapping_df = pd.DataFrame(sim_norm, index=styles, columns=accords)

print("=== 스타일-향조 소프트 매핑 행렬 (정규화 후) ===")
print(mapping_df.round(4))
print()

# ── 6. Top-2 향조 확인 ────────────────────────────────────────────────────────
print("=== 스타일별 Top-2 향조 ===")
for style in styles:
    top2 = mapping_df.loc[style].nlargest(2)
    print(f"  {style:10s}: {top2.index[0]} ({top2.iloc[0]:.3f}), "
          f"{top2.index[1]} ({top2.iloc[1]:.3f})")
print()

# ── 7. 저장 ───────────────────────────────────────────────────────────────────
mapping_df.to_csv('style_accord_mapping.csv')

mapping_dict = mapping_df.round(6).to_dict(orient='index')
with open('style_accord_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(mapping_dict, f, ensure_ascii=False, indent=2)

print("저장 완료: style_accord_mapping.csv / style_accord_mapping.json")

# ── 8. 시각화 ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- (A) 2D 좌표 공간 ---
ax = axes[0]
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.axvline(0, color='gray', lw=0.8, ls='--')

# 향수 휠 원 (참고용)
circle = plt.Circle((0, 0), r, color='lightblue', fill=False, ls='--', lw=1.2, alpha=0.6)
ax.add_patch(circle)

# 4대 계열 방향 표시
for family, angle, label in [
    ('Floral', 315, 'Floral\n(315°)'),
    ('Amber',   45, 'Amber\n(45°)'),
    ('Woody',  135, 'Woody\n(135°)'),
    ('Fresh',  225, 'Fresh\n(225°)'),
]:
    fx = 1.05 * np.cos(np.radians(angle))
    fy = 1.05 * np.sin(np.radians(angle))
    ax.annotate(label, (fx, fy), fontsize=7, color='steelblue',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', alpha=0.6))

# 향조 좌표
ACCORD_COLORS = {
    'floral': '#FF69B4', 'intense': '#8B0000',
    'natural': '#228B22', 'niche': '#4B0082', 'romantic': '#FF1493'
}
for accord, (ax_, ay_) in ACCORD_COORDS.items():
    angle = ACCORD_WHEEL[accord]
    ax.scatter(ax_, ay_, c=ACCORD_COLORS[accord], s=120, zorder=5, marker='D')
    ax.annotate(f'{accord}\n({angle}°)', (ax_, ay_),
                textcoords='offset points', xytext=(8, 6),
                fontsize=8, color=ACCORD_COLORS[accord], fontweight='bold')

# 스타일 좌표
STYLE_MARKERS = ['o','s','^','v','<','>']
STYLE_COLORS  = ['#E75480','#4682B4','#FF8C00','#2F4F4F','#696969','#8B4513']
for i, (style, (sx, sy)) in enumerate(STYLE_COORDS.items()):
    ax.scatter(sx, sy, c=STYLE_COLORS[i], s=200, zorder=6,
               marker=STYLE_MARKERS[i], edgecolors='black', lw=0.8)
    ax.annotate(style, (sx, sy), textcoords='offset points',
                xytext=(-10, 10), fontsize=9, fontweight='bold',
                color=STYLE_COLORS[i])

ax.set_xlabel('← Cool  |  Warm →', fontsize=10)
ax.set_ylabel('← Light  |  Heavy →', fontsize=10)
ax.set_title('Fashion Style & Fragrance Accord\n2D Coordinate Map\n'
             '(Edwards Wheel + Kobayashi Scale)', fontsize=10)
ax.grid(True, alpha=0.3)

# --- (B) 소프트 매핑 히트맵 ---
ax2 = axes[1]
im = ax2.imshow(mapping_df.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.4)
ax2.set_xticks(range(len(accords)))
ax2.set_xticklabels(accords, fontsize=11)
ax2.set_yticks(range(len(styles)))
ax2.set_yticklabels(styles, fontsize=11)
for i in range(len(styles)):
    for j in range(len(accords)):
        ax2.text(j, i, f'{mapping_df.values[i,j]:.3f}',
                 ha='center', va='center', fontsize=10,
                 color='black' if mapping_df.values[i,j] < 0.25 else 'white')
plt.colorbar(im, ax=ax2, shrink=0.8)
ax2.set_title('Style-Accord Soft Mapping Matrix\n'
              '(exp(-distance) normalized)', fontsize=10)

plt.tight_layout()
plt.savefig('style_accord_coordinate_map.png', dpi=150, bbox_inches='tight')
plt.close()
print("시각화 저장: style_accord_coordinate_map.png")

# ── 9. 근거 테이블 저장 ───────────────────────────────────────────────────────
rationale_rows = []

kobayashi_ref = {
    'Romantic': 'Pretty/Romantic 영역: Pink·Lavender → Warm+Light',
    'Casual':   'Casual/Natural 영역: Beige·Cream → Slightly Cool+Light',
    'Sporty':   'Dynamic/Sporty 영역: White·Bright Blue → Cool+Light',
    'Formal':   'Classic/Formal 영역: Navy·Burgundy → Warm+Heavy',
    'Street':   'Cool/Urban 영역: Black·Dark Grey → Cool+Heavy',
    'Vintage':  'Nostalgic/Vintage 영역: Brown·Mustard → Warm+Heavy',
}
for style, (sx, sy) in STYLE_COORDS.items():
    rationale_rows.append({
        'type': 'style', 'name': style, 'x': sx, 'y': sy,
        'source': 'Kobayashi Color Image Scale (1990)',
        'rationale': kobayashi_ref[style]
    })

accord_ref = {
    'floral':   'Floral 계열 (315°): Zarzo(2009) Feminine→Warm+Light 사분면',
    'intense':  'Amber 계열 (45°): Oriental/Spicy → Warm+Heavy 사분면',
    'natural':  'Fresh 계열 (225°): Aquatic/Green → Cool+Light 사분면',
    'niche':    'Dry Woods (157°): Woody~Fresh 경계, Cool+Heavy 쪽',
    'romantic': 'Soft Floral (337°): Floral~Amber 경계, Feminine+Warm',
}
for accord, angle in ACCORD_WHEEL.items():
    x, y = ACCORD_COORDS[accord]
    rationale_rows.append({
        'type': 'accord', 'name': accord, 'x': x, 'y': y,
        'source': 'Edwards Fragrance Wheel (2023) + Zarzo & Stanton (2009)',
        'rationale': accord_ref[accord]
    })

pd.DataFrame(rationale_rows).to_csv('style_coord_rationale.csv', index=False)
print("근거 테이블 저장: style_coord_rationale.csv")

총 샘플 수: 8701
스타일 분포:
style_class
Formal      1500
Casual      1500
Sporty      1500
Street      1500
Vintage     1500
Romantic    1201
Name: count, dtype: int64

=== 향조 좌표 (Edwards 향수 휠 각도 → 직교 좌표) ===
향조              각도        x        y
floral        315°   0.5657  -0.5657
intense        45°   0.5657   0.5657
natural       225°  -0.5657  -0.5657
niche         157°  -0.7364   0.3126
romantic      337°   0.7364  -0.3126

=== 스타일-향조 유클리드 거리 ===
          floral  intense  natural   niche  romantic
Romantic  0.1692   1.1774   0.9663  1.4575    0.4425
Casual    0.7834   1.2324   0.4015  0.8919    0.9405
Sporty    1.1675   1.5794   0.0741  0.8240    1.3495
Formal    1.1774   0.1692   1.5137  1.1722    0.9726
Street    1.4382   1.0785   0.9679  0.2520    1.4271
Vintage   1.0677   0.0929   1.5071  1.2505    0.8463

=== 스타일-향조 소프트 매핑 행렬 (정규화 후) ===
          floral  intense  natural   niche  romantic
Romantic  0.3506   0.1279   0.1580  0.0967    0.2668
Casual    0.2060   0.1315   0.3018  0.184